# Setup environment 

# Define Model

In [1]:
ord('0')

48

In [2]:
chr(48)

'0'

In [3]:
'0'.encode('utf-8')

b'0'

In [4]:
b'\x30'.decode('utf-8') # same as b'0'.decode('utf-8')

'0'

In [5]:
chr(0) # Null character - null string

'\x00'

In [34]:
print (ord(chr(48))) # 48 -> '0' -> 48

48


In [39]:
repr("0")

"'0'"

In [44]:
"this is a test" + chr(0) + "string"

'this is a test\x00string'

In [46]:
print ("this is a test" + chr(0) + "string")

this is a test string


In [47]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

In [53]:
decode_utf8_bytes_to_str_wrong(list('hello'.encode('utf-8')))

'hello'

# Dummy data training


In [3]:
# build vocabulary
vocabulary = {(i,):i for i in range(256)}
vocabulary[tuple(list('<|endoftext|>'.encode()))] = 256
next_vocab_item = 257

inv_vocabulary = {j: i for i, j in vocabulary.items()}

In [4]:
def get_counts(cache):
    "get current pair counts"
    counts = {}

    for byte_int_string in cache:
        # if len(byte_int_string) == 1:
        #     counts[key] = counts.get(key, 0) + 1
        #     continue
        for i in range(len(byte_int_string)-1):
            key = (byte_int_string[i], byte_int_string[i+1])
            counts[key] = counts.get(key, 0) + cache[byte_int_string]
    return counts

# counts = get_counts(cache)

In [ ]:
def merge(cache, counts, vocabulary, inv_vocabulary, merges):
    # add merged to vocab
    # change cache to reflect merges
    print()
    print (f"{counts=}")
    next_vocab_item = len(vocabulary)
    max_occur_pair = max(counts, key=lambda i: (counts[i], i))
    print (f"{max_occur_pair=} will be replaced with {next_vocab_item=}")
    
    new_cache = {}

    for byte_int_string in cache:
        print (byte_int_string)
        new_key = []
        i = 0
        while i < len(byte_int_string): 
            if i < len(byte_int_string)-1 and (byte_int_string[i], byte_int_string[i+1]) == max_occur_pair:
                new_key.append(next_vocab_item)
                i += 2
            else:
                new_key.append(byte_int_string[i])
                i += 1
        new_cache[tuple(new_key)] = cache[byte_int_string]

    vocabulary[max_occur_pair] = next_vocab_item
    inv_vocabulary[next_vocab_item] = max_occur_pair
    merges.append(max_occur_pair)

    return new_cache, merges

In [6]:
# build data cache with counts
data = "low low low low low lower lower widest widest widest newest newest newest newest newest newest"

# current tuple of subword token counts
cache = {}

for word in data.split(' '):
    byte_int_string = tuple(word.encode())
    cache[byte_int_string] = cache.get(byte_int_string, 0) + 1

cache

{(108, 111, 119): 5,
 (108, 111, 119, 101, 114): 2,
 (119, 105, 100, 101, 115, 116): 3,
 (110, 101, 119, 101, 115, 116): 6}

In [7]:
chr(101), chr(101)

('e', 'e')

In [8]:
NUM_MERGES = 6
merges = []
for i in range(NUM_MERGES):
    counts = get_counts(cache)
    cache, merges = merge(cache, counts, vocabulary, inv_vocabulary, merges)
    print (cache)


counts={(108, 111): 7, (111, 119): 7, (119, 101): 8, (101, 114): 2, (119, 105): 3, (105, 100): 3, (100, 101): 3, (101, 115): 9, (115, 116): 9, (110, 101): 6, (101, 119): 6}
max_occur_pair=(115, 116) will be replaced with next_vocab_item=257
(108, 111, 119)
(108, 111, 119, 101, 114)
(119, 105, 100, 101, 115, 116)
(110, 101, 119, 101, 115, 116)
{(108, 111, 119): 5, (108, 111, 119, 101, 114): 2, (119, 105, 100, 101, 257): 3, (110, 101, 119, 101, 257): 6}

counts={(108, 111): 7, (111, 119): 7, (119, 101): 8, (101, 114): 2, (119, 105): 3, (105, 100): 3, (100, 101): 3, (101, 257): 9, (110, 101): 6, (101, 119): 6}
max_occur_pair=(101, 257) will be replaced with next_vocab_item=258
(108, 111, 119)
(108, 111, 119, 101, 114)
(119, 105, 100, 101, 257)
(110, 101, 119, 101, 257)
{(108, 111, 119): 5, (108, 111, 119, 101, 114): 2, (119, 105, 100, 258): 3, (110, 101, 119, 258): 6}

counts={(108, 111): 7, (111, 119): 7, (119, 101): 2, (101, 114): 2, (119, 105): 3, (105, 100): 3, (100, 258): 3, (110, 1

In [9]:
merges

[(115, 116), (101, 257), (111, 119), (108, 259), (119, 258), (110, 101)]

In [10]:
merges
chr(115), chr(116), chr(101)

('s', 't', 'e')

# Tokenizer Implementation

In [11]:
import pickle
from collections.abc import Iterable, Iterator

class Tokenizer:
    def __init__(
        self, 
        vocab: dict[int, bytes], 
        merges: list[tuple[bytes, bytes]], 
        special_tokens: list[str] | None = None
    ):
        """
        Construct a tokenizer from a given vocabulary, list of merges, 
        and (optionally) a list of special tokens.
        """
        self.vocab = vocab
        self.merges = merges
        self.special_tokens = special_tokens

        self.input_vocab = {j:i for i, j in vocab.items()}
        

    @classmethod
    def from_files(
        cls, 
        vocab_filepath: str, 
        merges_filepath: str, 
        special_tokens: list[str] | None = None
    ) -> "Tokenizer":
        """
        Class method that constructs and returns a Tokenizer from a serialized 
        vocabulary and list of merges (in the same format that your BPE training 
        code output) and (optionally) a list of special tokens.
        """
        with open(vocab_filepath, 'rb') as vocab_file:
            vocab = pickle.load(vocab_file)
        
        with open(merges_filepath, 'rb') as merges_file:
            merges = pickle.load(merges_file)           
            
        return cls(vocab, merges, special_tokens)

    def _pretokenized_input(self, text: str) -> Iterable[str]:
        """
        Pre-tokenize input before encoding. Strategy same as while training the 
        tokenizer
        """
        for word in text.split(' '):
            yield word


    def encode(self, text: str) -> list[int]:
        """
        Encode an input text into a sequence of token IDs.
        TODO: This is non optimal - optimize it
        """
        # Gemini conversation - https://gemini.google.com/share/7366e82f856c
        encoded_string = []
        for word in self._pretokenized_input(text):
            byte_string_int =  list(word.encode('utf-8'))
            for merge in self.merges:
                merged_byte_string = []
                i = 0
                while i < len(byte_string_int):
                    if i < len(byte_string_int) - 1 and merge == (byte_string_int[i], byte_string_int[i+1]):
                        merged_byte_string.append(self.input_vocab[merge])
                        i += 2
                    else:
                        merged_byte_string.append(byte_string_int[i])
                        i += 1
                byte_string_int = merged_byte_string
            encoded_string.extend(byte_string_int)
        return encoded_string

        

    def encode_iterable(self, iterable: Iterable[str]) -> Iterator[int]:
        """
        Given an iterable of strings (e.g., a Python file handle), return a 
        generator that lazily yields token IDs. This is required for 
        memory-efficient tokenization of large files that we cannot directly 
        load into memory.
        """
        pass

    def decode(self, ids: list[int]) -> str:
        """
        Decode a sequence of token IDs into text.
        """
        bytes_arr = bytes(self._decode_helper(ids))
        return bytes_arr.decode('utf-8', errors='replace')



    def _decode_helper(self, ids: list[int]) -> list[int]: # return type is int representation of bytes
        # Note: self.vocab is actually output vocabulary
        output = []
        for token_id in ids:
            if len(self.vocab[token_id]) == 1:
                output.append(self.vocab[token_id][0])
            else:
                subtokens = self.vocab[token_id]
                output.extend(self._decode_helper(subtokens))
        return output


In [12]:
tokenizer = Tokenizer(
    inv_vocabulary, 
    merges,
    []
)

In [14]:
tokenizer.decode(tokenizer.encode("You are lower|<endoftext>|"))

'Youarelower|<endoftext>|'

# Rough

In [18]:
def print_counts(counts):
    "only works for alphabets - without special tokens"
    for count in sorted(counts, key=lambda i: counts[i], reverse=True):
        print ("".join(map(chr, count)))

print_counts(counts)

ne
eą
wi
id
dĂ
Ąe
er


In [19]:
"Buggy version"

def merge(cache, counts, vocabulary, inv_vocabulary):
    # add merged to vocab
    # change cache to reflect merges
    print()
    print (f"{counts=}")
    next_vocab_item = len(vocabulary)
    max_occur_pair = sorted(counts, key=lambda i: (counts[i], i), reverse=True)[0]
    print (f"{max_occur_pair=} will be replaced with {next_vocab_item=}")
    
    new_cache = {}

    for byte_int_string in cache:
        print (byte_int_string)
        # if len(byte_int_string) == 1:
        #     # V IMP - check if this is a bug. cache is used for counting pairs for merges. Single byte subword can be ignored as it will never be merged as we don't merge across subword boundaries
        #     # If this condition is removed - a single char in vocab will break new_cache[tuple(new_key)] = cache[byte_int_string]
        #     continue
        new_key = [byte_int_string[0]]
        i = 1
        while i < len(byte_int_string): 
            key = (byte_int_string[i-1], byte_int_string[i])
            if key != max_occur_pair:
                new_key.append(byte_int_string[i])
            else:
                new_key[i-1] = next_vocab_item
            i += 1
        new_cache[tuple(new_key)] = cache[byte_int_string]

    vocabulary[max_occur_pair] = next_vocab_item
    inv_vocabulary[next_vocab_item] = max_occur_pair

    return new_cache

In [20]:
"low low low low low" 
"lower lower" 
"widest widest widest" 
"newest newest newest newest newest newest"

import weakref


lo - 7
ow - 7
we - 8
er - 2
wi - 3
id - 3 
de - 3 
es - 9 
st - 9 
ne - 6
ew - 6

NameError: name 'lo' is not defined

# BPE

## Pre-tokenization

In [63]:
import os
import multiprocessing
import regex as re
from typing import BinaryIO, Iterator
from collections import Counter


NUM_CHUNKS = 4
DOC_SPLIT_TOKEN="<|endoftext|>"
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

# ==========================================
# MACRO-CHUNKING 
# ==========================================
def find_chunk_boundaries(
    file: BinaryIO,
    num_chunks: int,
    split_special_token: bytes
) -> list[int]:
    """
    Chunk the file into byte boundaries
    Each chunk ends with the split_special_token to ensure complete document inclusion
    Each chunk may contain zero or more split_special_tokens (one or more documents)

    May return fewer chunks if boundaries end up overlapping
    """
    
    file.seek(0, os.SEEK_END)
    file_size = file.tell()
    file.seek(0)

    chunk_size = file_size // num_chunks
    chunk_boundaries = [i * chunk_size for i in range(num_chunks+1)]
    chunk_boundaries[-1] = file_size  # to ensure that final chunk is not beyond file size

    mini_chunk_size = 4096 # bytes to read in memory to determine boundaries
    overlap_size = len(split_special_token)-1 # To ensure we can read special tokens at chunk boundaries

    # The starting and end boundaries of file remain fixed, we move others as per split token
    for idx in range(1, len(chunk_boundaries) - 1):
        initial_boundary = chunk_boundaries[idx]
        file.seek(initial_boundary)
        while True:
            mini_chunk = file.read(mini_chunk_size)
            if not mini_chunk:
                chunk_boundaries[idx] = file_size
                break

            found_at = mini_chunk.find(split_special_token)
            if found_at != -1:
                chunk_boundaries[idx] = initial_boundary + found_at + len(split_special_token) # keeps special token inside chunks
                break

            initial_boundary += mini_chunk_size - overlap_size
            file.seek(initial_boundary) # seek required as file handler is ahead by overlap size before this statement
    
    return sorted(list(set(chunk_boundaries)))

# ==========================================
# MICRO-CHUNKING 
# ==========================================
def stream_document_byte_range(
    filepath: str,
    start_byte: int,
    end_byte: int,
    split_special_token: bytes
) -> Iterator[str]:
    """
    Memory-efficient generator that only reads within assigned byte boundaries
    """
    buffer = b""
    chunk_size = 65536 # 64 * 1024 - 64KB

    bytes_to_read = end_byte - start_byte
    
    with open(filepath, 'rb') as file:
        file.seek(start_byte)

        while bytes_to_read > 0:
            current_chunk_size = min(chunk_size, bytes_to_read)

            raw_bytes = file.read(current_chunk_size)
            bytes_to_read -= len(raw_bytes)

            if not raw_bytes:
                break

            buffer += raw_bytes

            while split_special_token in buffer:
                chunk, buffer = buffer.split(split_special_token, 1)
                if chunk:
                    yield chunk.decode('utf-8', errors='ignore')
        
        if buffer:
            yield buffer.decode('utf-8', errors='ignore')

# ==========================================
# Pre-tokenization of File Chunks
# ==========================================
def pretokenize_file_byte_range(
    file_path: str,
    start_byte: int,
    end_byte: int,
):
    token_bytes = DOC_SPLIT_TOKEN.encode('utf-8')
    file_chunk_counts =  Counter()  

    for doc in stream_document_byte_range(file_path, start_byte, end_byte, token_bytes):
        for pattern_match in re.finditer(PAT, doc):
            byte_int_tuple = tuple(pattern_match.group(0).encode('utf-8'))
            file_chunk_counts[byte_int_tuple] += 1

    return file_chunk_counts
    

def pretokenize_file(
    file_path: str
) -> Counter[tuple[int]]:
    token_bytes = DOC_SPLIT_TOKEN.encode('utf-8')
    with open(file_path, 'rb') as  f:
        chunk_boundaries = find_chunk_boundaries(f, NUM_CHUNKS, token_bytes)

        chunks = zip(chunk_boundaries[:-1], chunk_boundaries[1:])
        tasks = [(file_path, start, end) for start, end in chunks]
        num_tasks = len(tasks)

        pretokenized_cache = Counter()
        with multiprocessing.Pool(processes=num_tasks) as pool:
            results = pool.starmap(pretokenize_file_byte_range, tasks)

            for chunk_counts in results:
                pretokenized_cache.update(chunk_counts)

    return pretokenized_cache

In [64]:
from tokenizer_utils import pretokenize_file

FILE_PATH = "data/TinyStoriesV2-GPT4-valid.txt"

pretokenzied_cache = pretokenize_file(FILE_PATH)

In [65]:
len(pretokenzied_cache)

13111

## Train BPE

In [66]:
"""
Requirements:

Output:
vocab: dict[int, bytes]
merges: list[tuple[bytes, bytes]] - ordered by the order of creation

Optimization: 
Don't iterate over all byte pairs to find most frequent pair. 
The only pairs that overlap with merge will have counts change
"""

"\nRequirements:\n\nOutput:\nvocab: dict[int, bytes]\nmerges: list[tuple[bytes, bytes]] - ordered by the order of creation\n\nOptimization: \nDon't iterate over all byte pairs to find most frequent pair. \nThe only pairs that overlap with merge will have counts change\n"

In [ ]:
from tokenizer.training_utils import get_counts, merge


# token_to_int = {numer.to_bytes(1): i for i in range(256)} - this is also possible but below representation is easier for debugging
token_to_int = {(i,): i for i in range(256)}
int_to_token = {j:i for i, j in token_to_int.items()}

In [68]:
NUM_MERGES = 6
merges = []
cache = pretokenzied_cache
for i in range(NUM_MERGES):
    counts = get_counts(cache)
    cache, merges = merge(cache, counts, vocabulary, merges)


In [69]:
merges

[(32, 116), (104, 101), (32, 97), (32, 115), (32, 119), (258, 258)]

In [70]:
def print_merges(merge_tuple):
    return chr(merge_tuple[0]), chr(merge_tuple[1])

list(map(print_merges, merges))

[(' ', 't'), ('h', 'e'), (' ', 'a'), (' ', 's'), (' ', 'w'), ('Ă', 'Ă')]